# درس ۱۰ - عامل‌های هوش مصنوعی در محیط تولید

در این درس با **الگوهای تولید** برای عامل‌های هوش مصنوعی با استفاده از چارچوب عامل مایکروسافت (پایتون) آشنا می‌شوید. موارد زیر پوشش داده شده‌اند:

- **قابلیت مشاهده** — افزودن زمان‌بندی و ثبت لاگ به تعاملات عامل
- **ارزیابی** — استفاده از یک عامل ارزیابی‌کننده برای سنجش کیفیت پاسخ‌ها
- **مدیریت هزینه** — استراتژی‌هایی برای بهینه‌سازی توکن‌ها و انتخاب مدل

سناریو یک **عامل سفر** است که به کاربران در برنامه‌ریزی سفر کمک می‌کند، با لایه‌های نظارت و ارزیابی روی آن.


## تنظیمات


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ملاحظات تولید

جابجایی عوامل هوش مصنوعی از نمونه‌های اولیه به تولید نیازمند توجه دقیق به سه ستون است:

1. **قابلیت مشاهده** — شما به دید واضحی از کاری که عامل انجام می‌دهد، مدت زمانی که صرف می‌کند و ابزارهایی که فراخوانی می‌کند نیاز دارید. بدون ردیابی و ثبت لاگ، اشکال‌زدایی مشکلات تولید تقریباً غیرممکن است.

2. **ارزیابی** — بررسی‌های خودکار کیفیت تضمین می‌کنند که پاسخ‌های عامل در طول زمان دقیق، کامل و مفید باقی بمانند. یک عامل ارزیاب می‌تواند پاسخ‌ها را بر اساس معیارهای تعریف‌شده امتیازدهی کند.

3. **مدیریت هزینه** — استفاده از توکن مستقیماً بر هزینه تأثیر می‌گذارد. استراتژی‌هایی مانند بهینه‌سازی پرسش، انتخاب مدل و کشینگ کمک می‌کنند هزینه‌ها را بدون قربانی کردن کیفیت کنترل کنند.


## ساخت یک عامل قابل مشاهده

ما ابزارهای سفر را تعریف می‌کنیم و فراخوانی عامل را با زمان‌بندی می‌پیچیم تا بتوانیم تأخیر را نظارت کنیم. در محیط تولید، شما با OpenTelemetry یا یک سیستم پیگیری مشابه یکپارچه خواهید شد.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## الگوهای ارزیابی

یک الگوی رایج در تولید استفاده از یک عامل دوم به عنوان **ارزیاب** است. ارزیاب پاسخ عامل اصلی را بر اساس معیارهای از پیش تعیین شده مانند کامل بودن، دقت و کمک‌کننده بودن نمره‌دهی می‌کند.

این امکان را فراهم می‌کند:
- دروازه‌های کیفیت خودکار قبل از رسیدن پاسخ‌ها به کاربران
- شناسایی پسرفت زمانی که پرامپت‌ها یا مدل‌ها تغییر می‌کنند
- نظارت مداوم بر عملکرد عامل در طول زمان


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## استراتژی‌های مدیریت هزینه

کنترل هزینه‌ها برای عوامل هوش مصنوعی تولیدی حیاتی است. در اینجا استراتژی‌های کلیدی آورده شده است:

| استراتژی | توضیحات |
|---|---|
| **بهینه‌سازی درخواست** | دستورالعمل‌های سیستم را مختصر نگه دارید. با حذف زمینه‌های تکراری، ورودی توکن‌ها را کاهش دهید. |
| **انتخاب مدل** | برای کارهای ساده مانند طبقه‌بندی یا استخراج از مدل‌های کوچکتر و ارزان‌تر (مثلاً GPT-4o-mini) استفاده کنید و مدل‌های بزرگ‌تر را برای استدلال‌های پیچیده‌تر رزرو کنید. |
| **کش کردن** | نتایج ابزارها و پرسش‌های پرتکرار را کش کنید تا از درخواست‌های مکرر API جلوگیری شود. |
| **بودجه توکن‌ها** | محدودیت `max_tokens` را تنظیم کنید تا پاسخ‌های غیرمنتظره طولانی نشود. |
| **گروه‌بندی** | جایی که ممکن است چندین پرسش کاربر را در یک درخواست API جمع کنید. |

در عمل، یک رویکرد طبقه‌بندی شده خوب عمل می‌کند: درخواست‌های ساده را به یک مدل سریع و ارزان ارسال کنید و فقط پرسش‌های پیچیده را به مدل قدرتمندتر انتقال دهید.


## خلاصه

در این درس شما یاد گرفتید چگونه:

1. **مشاهده‌پذیری را اضافه کنید** به تعاملات عامل با زمان‌بندی و ثبت رویدادها، که پایه‌ای برای ردیابی و نظارت فراهم می‌کند.
2. **پاسخ‌های عامل را به‌صورت خودکار ارزیابی کنید** با استفاده از یک عامل ارزیاب که کامل بودن، دقت و مفید بودن را نمره می‌دهد.
3. **هزینه‌ها را مدیریت کنید** از طریق بهینه‌سازی درخواست، انتخاب مدل، کشینگ و بودجه‌بندی توکن‌ها.

این الگوهای تولیدی کمک می‌کنند تا اطمینان حاصل شود که عوامل هوش مصنوعی شما قابل اعتماد، قابل اندازه‌گیری و مقرون‌به‌صرفه در مقیاس هستند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
